## SVM(Support Vector Machine)
### SVM
- 선을 구성하는 매개변수를 조정하여 데이터들을 구분하는 선을 찾고 이를 기반으로 패턴 인식
- 데이터와 패턴들과의 거리(<u>**마진**</u>)을 최대로 만드는 것이 가장 좋은 결과 도출

#### SVM 활용 BMI 측정 예측
- BMI = 몸무게 / 신장**2
- 18.5 <= BMI < 25 일 때, 표준 몸무게

#### BMI Data 생성
- Random 한 데이터 2만개 생성
- Column : 키, 몸무게, label, 정상체중, 비만

In [1]:
import numpy as np
np.random.seed(42)

In [2]:
def calc_bmi(h, w):
    result = ''
    bmi = w / (h/100)**2
    if bmi < 18.5:
        result = 'thin'
    elif bmi < 25:
        result = 'normal'
    else:
        result = 'fat'
        
    return result

In [3]:
calc_bmi(170, 50)

'thin'

In [6]:
# 출력 파일 준비
fp = open('../Data/bmi.csv', 'w', encoding='utf-8')
fp.write('height,weight,label\n')

# 무작위 데이터 생성
cnt = {"thin":0, "normal":0, "fat":0}

for _ in range(20000):
    h = np.random.randint(120, 200)
    w = np.random.randint(35, 90)
    label = calc_bmi(h, w)
    cnt[label] += 1
    fp.write(f"{h},{w},{label}\n")

fp.close()
print("OK", cnt)

OK {'thin': 5100, 'normal': 5632, 'fat': 9268}


#### BMI 공식을 사용하지 않고 BMI 예측

In [7]:
import pandas as pd

tbl = pd.read_csv("../Data/bmi.csv")
tbl.head()

,height,weight,label
0,171,63,normal
1,134,77,fat
2,191,55,thin
3,194,45,thin
4,143,37,thin


In [8]:
tbl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   height  20000 non-null  int64 
 1   weight  20000 non-null  int64 
 2   label   20000 non-null  object
dtypes: int64(2), object(1)
memory usage: 468.9+ KB


In [10]:
tbl.describe()
# 평균(mean) 값과 중앙값(50%) 을 비교하여 해당 DataSet이 정규분포를 이루고 있는지 확인

,height,weight
count,20000.000000,20000.000000
mean,159.182150,62.109900
std,23.046575,15.859048
min,120.000000,35.000000
25%,139.000000,48.000000
50%,159.000000,62.000000
75%,179.000000,76.000000
max,199.000000,89.000000


### 정규화
: 컬럼별로 최대값을 구해 최대값을 1로 만드는 작업

In [11]:
w = tbl.weight / tbl.weight.max()
h = tbl.height / tbl.height.max()

In [12]:
# Feature Data
wh = pd.concat([w, h], axis='columns')
wh.head()

,weight,height
0,0.707865,0.859296
1,0.865169,0.673367
2,0.617978,0.959799
3,0.505618,0.974874
4,0.415730,0.718593


In [13]:
from sklearn.model_selection import train_test_split

In [14]:
train_data, test_data, train_target, test_target = \
    train_test_split(
        wh,
        tbl.label,
        random_state=42,
        stratify=tbl.label
    )

In [15]:
from sklearn.svm import SVC

In [16]:
clf = SVC()

In [17]:
# 학습시키기
clf.fit(train_data, train_target)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [18]:
# 검증
print('Train : ', clf.score(train_data, train_target))
print('Train : ', clf.score(test_data, test_target))

Train :  0.9968
Train :  0.9962
